In [1]:
!pip install clickhouse-connect pandas numpy


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import uuid
import clickhouse_connect

# Danh sách sensor mới mở rộng
sensor_ids = [
    "3276359", "3276360", "3276361", "3276362",
    "3276363", "3276364", "3276365", "3276366",
    "3276367", "3276368"
]

# Cấu hình sensor mới
sensor_info = {
    "3276359": {"area": "Quận 1", "location_name": "Hồ Chí Minh", "latitude": 10.776, "longitude": 106.7, "owner_name": "Hoaze", "provider": "Hoaze", "unit": "µg/m3"},
    "3276360": {"area": "Quận 2", "location_name": "Hồ Chí Minh", "latitude": 10.785, "longitude": 106.73, "owner_name": "EnviroTech", "provider": "ProviderA", "unit": "µg/m3"},
    "3276361": {"area": "Quận 3", "location_name": "Hồ Chí Minh", "latitude": 10.779, "longitude": 106.695, "owner_name": "SensorCorp", "provider": "ProviderB", "unit": "µg/m3"},
    "3276362": {"area": "Quận 4", "location_name": "Hồ Chí Minh", "latitude": 10.767, "longitude": 106.7, "owner_name": "EnviroTech", "provider": "ProviderC", "unit": "µg/m3"},
    
    # Thêm sensor mới ở các quận khác
    "3276363": {"area": "Quận 5", "location_name": "Hồ Chí Minh", "latitude": 10.755, "longitude": 106.66, "owner_name": "AirMonitor", "provider": "ProviderD", "unit": "µg/m3"},
    "3276364": {"area": "Quận 6", "location_name": "Hồ Chí Minh", "latitude": 10.746, "longitude": 106.65, "owner_name": "CleanAir", "provider": "ProviderE", "unit": "µg/m3"},
    "3276365": {"area": "Quận 7", "location_name": "Hồ Chí Minh", "latitude": 10.74, "longitude": 106.72, "owner_name": "SensorTech", "provider": "ProviderF", "unit": "µg/m3"},
    "3276366": {"area": "Quận 8", "location_name": "Hồ Chí Minh", "latitude": 10.73, "longitude": 106.65, "owner_name": "EnviroTech", "provider": "ProviderG", "unit": "µg/m3"},
    
    # Sensor trải rộng ra các tỉnh lân cận
    "3276367": {"area": "Thủ Đức", "location_name": "Hồ Chí Minh", "latitude": 10.86, "longitude": 106.78, "owner_name": "Hoaze", "provider": "VNPT", "unit": "µg/m3"},
    "3276368": {"area": "Biên Hòa", "location_name": "Đồng Nai", "latitude": 10.95, "longitude": 106.82, "owner_name": "EnviroTech", "provider": "ProviderH", "unit": "µg/m3"}
}

# Thêm sensor khu vực Hà Nội
sensor_ids.extend([
    "3276369", "3276370", "3276371", "3276372"
])

sensor_info.update({
    "3276369": {"area": "Ba Đình", "location_name": "Hà Nội", "latitude": 21.033, "longitude": 105.814, "owner_name": "AirMonitor", "provider": "ProviderX", "unit": "µg/m3"},
    "3276370": {"area": "Hoàn Kiếm", "location_name": "Hà Nội", "latitude": 21.028, "longitude": 105.85, "owner_name": "CleanAir", "provider": "ProviderY", "unit": "µg/m3"},
    "3276371": {"area": "Cầu Giấy", "location_name": "Hà Nội", "latitude": 21.04, "longitude": 105.79, "owner_name": "VNPT", "provider": "VNPT", "unit": "µg/m3"},
    "3276372": {"area": "Nam Từ Liêm", "location_name": "Hà Nội", "latitude": 21.00, "longitude": 105.77, "owner_name": "EnviroTech", "provider": "ProviderZ", "unit": "µg/m3"}
})

# Thời gian sinh dữ liệu (Đã thêm năm 2025, 2024)
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)
hours_per_day = 24

# Hàm sinh dữ liệu 1 sensor 1 giờ
def generate_sensor_data(sensor_id, dt):
    meta = sensor_info[sensor_id]

    co = np.random.uniform(0.1, 10)
    no2 = np.random.uniform(5, 50)
    so2 = np.random.uniform(5, 50)
    pm25 = np.random.uniform(10, 50)
    pm10 = np.random.uniform(10, 50)
    o3 = np.random.uniform(5, 50)

    aqi_co = co * 1.5
    aqi_no2 = no2 * 2
    aqi_so2 = so2 * 2
    aqi_pm25 = pm25
    aqi_pm10 = pm10
    aqi_o3 = o3 * 1.2
    aqi_total = max(aqi_co, aqi_no2, aqi_so2, aqi_pm25, aqi_pm10, aqi_o3)
    pollutants = ['co', 'no2', 'so2', 'pm25', 'pm10', 'o3']
    aqi_values = [aqi_co, aqi_no2, aqi_pm25, aqi_pm10, aqi_o3]
    main_pollutant = pollutants[np.argmax(aqi_values)]

    return {
        'sensor_id': int(sensor_id),
        'area': meta['area'],
        'location_name': meta['location_name'],
        'datetimeLocal': dt,
        'timezone': 'Asia/Ho_Chi_Minh',
        'latitude': meta['latitude'],
        'longitude': meta['longitude'],
        'owner_name': meta['owner_name'],
        'provider': meta['provider'],
        'co_avg': round(co, 2),
        'no2_avg': round(no2, 2),
        'so2_avg': round(so2, 2),
        'pm25_avg': round(pm25, 2),
        'pm10_avg': round(pm10, 2),
        'o3_avg': round(o3, 2),
        'o3_8h_avg': round(o3, 2),
        'aqi_co': round(aqi_co, 2),
        'aqi_no2': round(aqi_no2, 2),
        'aqi_so2': round(aqi_so2, 2),
        'aqi_pm25': round(aqi_pm25, 2),
        'aqi_pm10': round(aqi_pm10, 2),
        'aqi_o3': round(aqi_o3, 2),
        'aqi_total': round(aqi_total, 2),
        'main_pollutant': main_pollutant,
        'unit': meta['unit'],
        'inserted_at': datetime.now()
    }

# Tạo dữ liệu cho nhiều ngày và nhiều sensor
data = []
current_date = start_date
while current_date <= end_date:
    for h in range(hours_per_day):
        dt = current_date + timedelta(hours=h)
        for sensor_id in sensor_ids:
            data.append(generate_sensor_data(sensor_id, dt))
    current_date += timedelta(days=1)

df = pd.DataFrame(data)
print(f"✅ Tạo {len(df)} bản ghi từ {start_date.date()} đến {end_date.date()} cho {len(sensor_ids)} sensor.")

# Ghi vào ClickHouse
client = clickhouse_connect.get_client(
    host='localhost',
    user='default',
    password='admin',
    database='hoaze',
    secure=False  # Set to True if using HTTPS
)

client.insert_df('air_quality_analytics', df)
print("✅ Đã chèn dữ liệu vào ClickHouse thành công!")


Unable to connect optimized C data functions [No module named 'clickhouse_connect.driverc.buffer'], falling back to pure Python


✅ Tạo 122640 bản ghi từ 2025-01-01 đến 2025-12-31 cho 14 sensor.
✅ Đã chèn dữ liệu vào ClickHouse thành công!
